# 📊 Aurora Forecasts — Yearly Price Processing

> **Author:** `<!-- your name -->`  · **Date:** `<!-- date -->`  · **Version:** 1.0

---

## Abstract

This notebook consolidates and transforms yearly Aurora Energy Research forecast data into a publication-ready price matrix. It reads raw **technology** (capture prices) and **system** (commodity prices) Parquet files, applies country-specific inflation adjustments to align every scenario to a common real-price base year, and pivots the result into the wide tracker format consumed by the downstream Excel price tracker.

Running this notebook produces both a Parquet snapshot (`aurora_prices_melted_default_currency_1y.parquet`) and an Excel tracker file (`prices_tracker.xlsx`), which is also uploaded to the shared SharePoint library.

---

## 🗺️ Notebook Map

| Step | Section | Description |
|---|---|---|
| 1 | Imports | Load `pandas` and project packages |
| 2 | Paths & Parameters | Define I/O paths, target currency year, and commodity list |
| 3 | Data Loading | Read raw Parquet files for technology and system scenarios |
| 4 | Technology Price Preprocessing | Filter capture prices; map curtailment type and scope |
| 5 | Inflation Data & Adjuster | Load inflation Excel; initialise `InflationAdjuster` |
| 6 | Country/Region Enrichment | Append human-readable country and region labels |
| 7 | Inflation Adjustment | Convert all prices to a common real base year |
| 8 | Merge & Format | Combine commodity and technology prices into one DataFrame |
| 9 | Parquet Export | Persist the melted price dataset |
| 10 | Tracker Format | Pivot to wide format; add release metadata |
| 11 | Excel & SharePoint Export | Write tracker Excel; upload to SharePoint |

> ⚙️ **Base currency year:** `currency_year = 2025`
> 💾 **Primary output:** `outputs/trackers/prices_tracker.xlsx`

## 📦 1 · Imports & Environment Setup

Load the standard data-manipulation library needed throughout this notebook.

| | |
|---|---|
| **Output** | `pd` (pandas) alias ready for use |

In [1]:
# Standard tabular data library — used throughout for DataFrame operations
import pandas as pd

In [2]:
# --- Output path ---
# Excel tracker consumed by the downstream price-reporting workflow
tracker_path = '../../../outputs/trackers/prices_tracker.xlsx'

# --- Raw data paths (Parquet) produced by notebook 1a ---
path_technology = '../../../data/aurora/aurora2_technology_scenarios_ES_default_currency_1y.parquet'
path_system     = '../../../data/aurora/aurora2_system_scenarios_ES_default_currency_1y.parquet'

# --- Inflation reference data ---
inflation_path = '../../../data/finance/inflation.xlsx'
inflation_all_countries_path = '../../../outputs/trackers/inflation_mapping_all_countries.xlsx'

# --- Intermediate processed data ---
path_prices_melted = '../../../data/processed/aurora_prices_melted_default_currency_1y.parquet'

# --- SharePoint / MS Graph config ---
ms_config_path = '../../../config/ms_config.yaml'

In [3]:
# Target real-price year for all inflation adjustments.
# All scenario prices will be expressed in real prices of this year.
# currency_year = datetime.now().year  # dynamic alternative — currently pinned for reproducibility
currency_year = 2025

# Aurora system variable names for commodity prices.
# Update this list if Aurora adds new commodity price types to the API.
commodity_list = ['Baseload price', 'Carbon price', 'Gas price', 'Coal price']

In [4]:
# Load raw yearly scenario Parquet files produced by notebook 1a.
df_technology2 = pd.read_parquet(path_technology)  # technology-level capture prices
df_system2     = pd.read_parquet(path_system)       # system-level commodity + market prices

In [5]:
# Sanity-check: confirm the dataset contains no negative-price-hour variables.
# Aurora occasionally adds 'neg*' series; if found they must be handled explicitly before pivoting.
df_technology2[df_technology2['type'].str.lower().str.contains('neg')]['type'].unique()

array([], dtype=object)

In [6]:
# Filter technology data to capture prices only for the three core renewable groups.
# BTM (Behind-the-Meter) solar is excluded — only front-of-meter generation is relevant for PPA economics.
df_prices = df_technology2[
     (df_technology2['type'].str.lower().str.contains('price'))        # keep price rows only
    &
    (df_technology2['Group'].isin(['Solar', 'Onshore wind', 'Offshore wind']))
    &
    (~(df_technology2['Subgroup'] == 'BTM solar PV'))                  # exclude behind-the-meter solar
    ]

print(f"{df_technology2['type'].unique()}")
df_prices.reset_index(drop=True, inplace=True)

['Capacity' 'Generation' 'Marginal technology' 'New build capacity'
 'Uncurtailed capture price' 'Curtailed capture price (M0)'
 'Capture price curtailed below zero' 'Curtailment rate below zero'
 'Capture price curtailed - fleet wide' 'Curtailment rate - fleet wide'
 'M0 reference price']


In [7]:
# Map each price type to its curtailment classification ('Uncurtailed' or 'Curtailed below zero').
# The mapping dict is maintained in aurora_forecasts.processing.dicts.
from aurora_forecasts.processing.dicts import curtailment_type_dict

df_prices.loc[:, "curtailed_uncurtailed"] = df_prices['type'].map(curtailment_type_dict)
df_prices.loc[:, "scope"] = "Capture"  # all technology rows in df_prices represent capture prices

# Guard: raise immediately if any price type is not covered by the mapping dict —
# an unmapped type would silently produce NaN in downstream aggregations.
if df_prices[df_prices['curtailed_uncurtailed'].isnull()].empty is False:
    raise ValueError("There are some price types that have not been mapped to curtailed/uncurtailed.")
else: 
    pass

C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\1782371364.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prices.loc[:, "curtailed_uncurtailed"] = df_prices['type'].map(curtailment_type_dict)
C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\1782371364.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prices.loc[:, "scope"] = "Capture"  # all technology rows in df_prices represent capture prices


In [8]:
# Load the inflation reference table from Excel.
# Columns 2015–2062 contain annual inflation rates (%) and/or price index values by country and type.
inflation_figures_df = pd.read_excel(inflation_path, sheet_name='Sheet1')
max_year = df_prices['year'].max()
if max_year > 2062:
    raise ValueError(f"The maximum year in the price scenarios ({max_year}) exceeds the maximum year in the inflation reference data (2062). Please update the inflation reference data to include years up to {max_year}.")

# Cast year columns to float to ensure numeric operations work correctly
for column in range(2015, max_year + 1):
    inflation_figures_df[column] = inflation_figures_df[column].astype(float)
    
print(inflation_figures_df['Date'].dtype)  # should stay 'object' (strings) throughout

object


In [9]:
inf_df = inflation_figures_df.copy()
inf_df['normalized_date'] = pd.to_datetime(inf_df['Date'], format="%Y-%b")
latest_date = pd.to_datetime(inf_df['Date'], format="%Y-%b").max().strftime("%Y-%b")

latest_date

'2026-Apr'

In [ ]:
# Initialise the InflationAdjuster with the full inflation reference table.
# Country and date filter are omitted here — they are passed per-call in transform_to_real_prices_yearly,
# allowing the same adjuster instance to handle multiple countries in one notebook run.
from common_libs.finance.finance import InflationAdjuster

adjuster = InflationAdjuster(
    inflation_df=inflation_figures_df,
    # country='Spain',       # could set a default; currently overridden per-call
    # date_filter='2025-Apr' # latest available inflation vintage
)

'2026-Apr'

In [11]:
adjuster.create_conversion_table_all_countries(base_year=currency_year, export_path=inflation_all_countries_path)

,release,country,metric_type,1980,1981,1982,1983,1984,1985,1986,...,2053,2054,2055,2056,2057,2058,2059,2060,2061,2062
0,2025-Apr,France,Inflation (annual % change),13.100000,13.300000,12.000000,9.500000,7.700000,5.800000,2.500000,...,1.900000,1.900000,1.900000,1.900000,1.900000,1.900000,1.900000,1.900000,1.900000,1.900000
1,2025-Apr,France,Index to real 2025,3.509959,3.097934,2.766012,2.526039,2.345440,2.216862,2.162792,...,0.592695,0.581643,0.570798,0.560155,0.549711,0.539461,0.529402,0.519531,0.509844,0.500338
2,2025-Apr,France,Index to nominal,0.284904,0.322796,0.361531,0.395877,0.426359,0.451088,0.462365,...,1.687209,1.719266,1.751933,1.785219,1.819138,1.853702,1.888922,1.924812,1.961383,1.998650
3,2025-Apr,Germany,Inflation (annual % change),5.400000,6.300000,5.300000,3.300000,2.400000,2.100000,-0.100000,...,2.200000,2.200000,2.200000,2.200000,2.200000,2.200000,2.200000,2.200000,2.200000,2.200000
4,2025-Apr,Germany,Index to real 2025,2.708049,2.547553,2.419328,2.342041,2.287149,2.240107,2.242350,...,0.547462,0.535677,0.524146,0.512863,0.501823,0.491020,0.480450,0.470108,0.459988,0.450086
5,2025-Apr,Germany,Index to nominal,0.369270,0.392534,0.413338,0.426978,0.437225,0.446407,0.445961,...,1.826611,1.866797,1.907866,1.949839,1.992736,2.036576,2.081380,2.127171,2.173969,2.221796
6,2025-Apr,Ireland,Inflation (annual % change),18.300000,20.200000,17.200000,10.400000,8.600000,5.500000,3.000000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000
7,2025-Apr,Ireland,Index to real 2025,4.226689,3.516380,3.000325,2.717685,2.502473,2.372012,2.302924,...,0.577201,0.565883,0.554787,0.543909,0.533244,0.522788,0.512538,0.502488,0.492635,0.482976
8,2025-Apr,Ireland,Index to nominal,0.236592,0.284383,0.333297,0.367960,0.399605,0.421583,0.434231,...,1.732500,1.767150,1.802493,1.838543,1.875314,1.912820,1.951076,1.990098,2.029900,2.070498
9,2025-Apr,Italy,Inflation (annual % change),21.800000,19.500000,16.500000,14.700000,10.700000,9.000000,5.800000,...,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000,2.000000


In [12]:
# Enrich df_prices with human-readable country and region labels by mapping region_code.
# The InflationAdjuster matches on country name (e.g. 'Spain'), not on region code (e.g. 'esp').
from aurora_forecasts.processing.dicts import country_tracker_map, region_tracker_map

df_prices.loc[:, 'country'] = df_prices.loc[:, 'region_code'].map(country_tracker_map)
df_prices.loc[:, 'region']  = df_prices.loc[:, 'region_code'].map(region_tracker_map)

# Guard: any unmapped region_code would silently produce NaN, corrupting the inflation adjustment
if (df_prices[df_prices['country'].isnull()].empty or df_prices[df_prices['region'].isnull()].empty) is False:
    raise ValueError("There are some price types that have not been mapped to curtailed/uncurtailed.")
else: 
    pass

df_prices

C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\260053826.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prices.loc[:, 'country'] = df_prices.loc[:, 'region_code'].map(country_tracker_map)
C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\260053826.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_prices.loc[:, 'region']  = df_prices.loc[:, 'region_code'].map(region_tracker_map)


,year,Group,Subgroup,type,value,unit,name,currency,sensitivity,region_code,curtailed_uncurtailed,scope,country,region
0,2021,Onshore wind,Onshore wind,Uncurtailed capture price,50.03,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Uncurtailed,Capture,Spain,Spain
1,2021,Solar,Solar PV,Uncurtailed capture price,49.53,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Uncurtailed,Capture,Spain,Spain
2,2022,Onshore wind,Onshore wind,Uncurtailed capture price,54.55,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Uncurtailed,Capture,Spain,Spain
3,2022,Solar,Solar PV,Uncurtailed capture price,51.87,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Uncurtailed,Capture,Spain,Spain
4,2023,Onshore wind,Onshore wind,Uncurtailed capture price,55.21,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Uncurtailed,Capture,Spain,Spain
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
186504,2060,Solar,Fixed solar PV,Capture price curtailed below zero,27.40,EUR/MWh,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Curtailed below zero,Capture,Portugal,Portugal
186505,2060,Solar,Fixed solar PV,Uncurtailed capture price,27.36,EUR/MWh,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Uncurtailed,Capture,Portugal,Portugal
186506,2060,Solar,Tracking solar PV,Capture price curtailed - fleet wide,27.39,EUR/MWh,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Curtailed,Capture,Portugal,Portugal
186507,2060,Solar,Tracking solar PV,Capture price curtailed below zero,27.27,EUR/MWh,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Curtailed below zero,Capture,Portugal,Portugal


In [13]:
# Filter system scenarios to the four commodity price variables.
# Apply the same country/region enrichment so the InflationAdjuster can match on country name.
df_commodities = df_system2[df_system2['variable'].isin(commodity_list)]

df_commodities['region']  = df_commodities['region_code'].map(region_tracker_map)
df_commodities['country'] = df_commodities['region_code'].map(country_tracker_map)

# Guard: unmapped region codes would produce NaN and silently corrupt the inflation adjustment
if (df_commodities[df_commodities['country'].isnull()].empty or df_commodities[df_commodities['region'].isnull()].empty) is False:
    raise ValueError("There are some commodity price types that have not been mapped to country/region.")
else: 
    pass

df_commodities

C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\1247135968.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_commodities['region']  = df_commodities['region_code'].map(region_tracker_map)
C:\Users\mpy\AppData\Local\Temp\ipykernel_61680\1247135968.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_commodities['country'] = df_commodities['region_code'].map(country_tracker_map)


,year,variable,value,units,name,currency,sensitivity,region_code,region,country
0,2021,Baseload price,54.36,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
1,2022,Baseload price,59.13,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
2,2023,Baseload price,60.37,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
3,2024,Baseload price,61.85,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
4,2025,Baseload price,63.52,EUR/MWh,Iberia October 2020 (High),eur2019,high,esp,Spain,Spain
...,...,...,...,...,...,...,...,...,...,...
597971,2056,Coal price,64.47,EUR/tonne,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
597972,2057,Coal price,63.72,EUR/tonne,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
597973,2058,Coal price,63.03,EUR/tonne,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal
597974,2059,Coal price,62.45,EUR/tonne,Iberia Q2 26 (Low Extended GenTax),eur2025,lowextendedgenerationtax,prt,Portugal,Portugal


In [14]:
df = df_commodities.copy()

df[df['currency'] == 'eur2025jan']['name'].unique()
df_commodities['currency'].unique()

array(['eur2019', 'gbp2019', 'pln2019', 'gbp2020', 'eur2020', 'pln2020',
       'gbp2021', 'eur2021', 'pln2021', 'eur2022', 'pln2022', 'gbp2022',
       'eur2023', 'pln2023', 'gbp2023', 'eur2024', 'pln2024', 'gbp2024',
       'eur2025', 'eur2025jan', 'pln2025', 'gbp2025'], dtype=object)

In [15]:
# Apply country-specific inflation adjustment to both datasets.
# transform_to_real_prices_yearly converts each price from its native Aurora real-currency year
# (stored in the 'currency' column, e.g. 'eur2022') to real prices expressed in currency_year (2025).
df_commodities_adj = adjuster.transform_to_real_prices_yearly(
    df_commodities[df_commodities['variable'].isin(commodity_list)],
    base_year=currency_year,
    year_col='year',
    price_cols=['value'],
    country_col='country',
)

df_technology_prices_adj = adjuster.transform_to_real_prices_yearly(
    df=df_prices, 
    base_year=currency_year,
    year_col='year',
    price_cols=['value'],
    country_col='country', 
)

In [16]:
# Harmonise the schemas of both adjusted DataFrames before concatenating.

# --- Commodity adjustments ---
df_commodities_adj_format = df_commodities_adj.copy()
df_commodities_adj_format['curtailed_uncurtailed'] = 'N/A'  # commodities have no curtailment concept

from aurora_forecasts.processing.dicts import commodity_scope_dict

# Map each commodity to its thematic scope (e.g. 'Commodity', 'Baseload')
df_commodities_adj_format['scope'] = df_commodities_adj_format['variable'].map(commodity_scope_dict)
if df_commodities_adj_format['scope'].isnull().any():
    raise ValueError("There are some commodity types that have not been mapped to 'scope'.")
else:
    pass

# --- Technology capture price adjustments ---
df_prices_format_adjusted = df_technology_prices_adj.copy()
# Drop redundant columns and rename to align with the commodity schema
df_prices_format_adjusted.drop(columns=['Group', 'type'], inplace=True)
df_prices_format_adjusted.rename(columns={'Subgroup': 'variable', 'unit': 'units'}, inplace=True)

# Concatenate both datasets into a single unified long-format price DataFrame
df_concat = pd.concat([df_commodities_adj_format, df_prices_format_adjusted], ignore_index=True)

df_concat.sort_values(by=['name', 'year', 'variable', 'sensitivity', 'currency', 'region_code'], inplace=True)
df_concat.reset_index(drop=True, inplace=True)

In [17]:
# Export the merged, inflation-adjusted price dataset as Parquet.
# Drop the intermediate inflation columns — downstream consumers only need the original 'value' field.
df_export = df_concat.copy()
df_export.drop(columns=['nominal_value', f'real_value_{currency_year}'], inplace=True)
df_export.to_parquet(path_prices_melted, index=False)

In [ ]:
# Build the wide-format tracker DataFrame consumed by the price tracker Excel template.
# Steps: (1) filter out deprecated scenario names, (2) parse release metadata from the name string,
# (3) standardise variable names, (4) pivot from long → wide (year as columns).

df_concat_fil = df_concat.copy()

# Remove specific named scenarios that are incomplete or have been superseded
df_concat_fil = df_concat[
    (~df_concat['name'].isin(['Iberia Jan 23 (Low - No Generation Tax)', 'Great Britain Jul 22 Locational Balancing (Central)']))
    ].reset_index(drop=True)

from aurora_forecasts.utils.forecast_release import parse_release_info  # bring in helper to parse release metadata from the release name

# Derive structured release info for each row so we can pull out quarter, month, and year later in the pipeline
df_concat_fil['release'] = df_concat_fil['name'].apply(parse_release_info)

# Extract the quarter number to support filtering/grouping by release period
df_concat_fil["release_quarter"] = df_concat_fil["release"].apply(lambda x: x.quarter)

# Pull the release month, which may be needed for tracker formatting or consistency checks
df_concat_fil["release_month"]   = df_concat_fil["release"].apply(lambda x: x.month)

# Pull the string month, which will be needed afterwards for tracker formatting
df_concat_fil['string_month']    = df_concat_fil["release"].apply(lambda x: x.month_str)

# Capture the release year so that every row carries the release publication year alongside the actual data year
df_concat_fil["release_year"]    = df_concat_fil["release"].apply(lambda x: x.year)

# Build a composite release identifier (e.g. '2025-Oct') used for grouping rows in the tracker
df_concat_fil['release_string'] = df_concat_fil['release_year'].astype(str) + '-' + df_concat_fil['string_month']

df_concat_fil.sort_values(by=['name', 'year', 'variable'], inplace=True)
df_concat_fil[df_concat_fil['variable'] == 'Baseload price'].shape[0]

df_concat_fil['Source'] = 'Aurora'
df_concat_fil['Scenario'] = df_concat_fil['sensitivity'].str.capitalize()

# Standardise technology label: some Aurora releases use 'Solar', others use 'Solar PV'
df_concat_fil['variable'] = df_concat_fil['variable'].str.replace('Solar', 'Solar PV', regex=False)

# Replace 'Baseload price' with 'N/A' to match the tracker's variable naming convention
df_concat_fil['variable'] = df_concat_fil['variable'].replace({'Baseload price': 'N/A'})

if df_concat_fil['variable'].isnull().any():
    raise ValueError("There are some variable types that have not been mapped to tracker variable names.")
else:
    pass

# Strip the year suffix from the currency code (e.g. 'eur2023' → 'EUR') for cleaner display
df_concat_fil['currency'] = df_concat_fil['currency'].str.slice(0, -4).str.upper()

# Define the multi-level index for the pivot — each combination identifies a unique tracker row
pivot_index = ['region', 'Source', 'release_year', 'string_month', 'release_string', 'Scenario', 'curtailed_uncurtailed',  
    'scope', 'variable', 'currency', 'currency_year']

# Pivot from long (one row per forecast year) to wide (one column per forecast year)
df_concat_fil_pivotted = df_concat_fil.pivot(
    index=pivot_index,
    columns='year',
    values='value'
).reset_index()

# Pad missing year columns from 2016 onwards to ensure a uniform schema across all scenario vintages
df_concat_fil_pivotted_columns = df_concat_fil_pivotted.columns.tolist()
for year in range(2016, df_concat_fil['year'].max()+1):
    if year not in df_concat_fil_pivotted_columns:
        df_concat_fil_pivotted[year] = pd.NA

# Reorder columns: metadata index followed by ascending forecast years
df_concat_fil_pivotted = df_concat_fil_pivotted.reindex(
    columns= pivot_index + list(range(2016, df_concat_fil['year'].max()+1))
)

df_concat_fil_pivotted.sort_values(by=['region', 'Source', 'release_year', 'string_month', 'Scenario', 'curtailed_uncurtailed',
    'scope', 'variable', 'currency', 'currency_year'], inplace=True)

df_concat_fil_pivotted

year,region,Source,release_year,string_month,release_string,Scenario,curtailed_uncurtailed,scope,variable,currency,...,2051,2052,2053,2054,2055,2056,2057,2058,2059,2060
0,France,Aurora,2020,Oct,2020-Oct,Central,N/A,Baseload,N/A,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,France,Aurora,2020,Oct,2020-Oct,Central,N/A,Commodity,Carbon price,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,France,Aurora,2020,Oct,2020-Oct,Central,N/A,Commodity,Coal price,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,France,Aurora,2020,Oct,2020-Oct,Central,N/A,Commodity,Gas price,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,France,Aurora,2020,Oct,2020-Oct,Central,Uncurtailed,Capture,Offshore wind,EUR,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9900,Spain,Aurora,2026,Jan,2026-Jan,Lowextendedgenerationtax,N/A,Commodity,Gas price,EUR,...,18.58,18.48,18.41,18.27,18.13,17.93,17.81,17.62,17.79,17.75
9901,Spain,Aurora,2026,Jan,2026-Jan,Lowextendedgenerationtax,Uncurtailed,Capture,Fixed solar PV,EUR,...,28.83,28.54,28.90,29.76,30.09,29.63,30.03,30.03,30.32,29.95
9902,Spain,Aurora,2026,Jan,2026-Jan,Lowextendedgenerationtax,Uncurtailed,Capture,Floating offshore wind,EUR,...,40.67,40.07,40.15,40.71,40.79,40.22,40.69,40.65,41.21,41.15
9903,Spain,Aurora,2026,Jan,2026-Jan,Lowextendedgenerationtax,Uncurtailed,Capture,Onshore wind,EUR,...,37.64,36.90,36.97,37.49,37.49,36.93,37.39,37.33,37.86,37.81


In [19]:
# Write the wide-format tracker to Excel.
# This file is the primary deliverable consumed by the price-tracking dashboard and reporting templates.
df_concat_fil_pivotted.to_excel(tracker_path, index=False)

In [20]:
# Create a tracker of the different data that appears in the API

df_drivers = df_concat_fil_pivotted.copy()

index = ['Scenario', 'curtailed_uncurtailed','scope', 'variable']
release_string_list = df_drivers['release_string'].unique().tolist()

release_dict = {}

for release_string in release_string_list:
    df_drivers_pivot = df_drivers[df_drivers['release_string'] == release_string].pivot(
        index=index,
        columns='region',
        values='release_string'
    )
    release_dict[release_string] = df_drivers_pivot

df_drivers_pivot

region                                                                             France  \
Scenario                 curtailed_uncurtailed scope     variable                           
Central                  Curtailed             Capture   Fixed solar PV          2026-Jan   
                                                         Floating offshore wind       NaN   
                                                         Offshore wind                NaN   
                                                         Onshore wind            2026-Jan   
                                                         Tracking solar PV       2026-Jan   
...                                                                                   ...   
Lowextendedgenerationtax N/A                   Commodity Gas price                    NaN   
                         Uncurtailed           Capture   Fixed solar PV               NaN   
                                                         Floating offshore wind       NaN   
                                                         Onshore wind                 NaN   
                                                         Tracking solar PV            NaN   

region                                                                            GB  \
Scenario                 curtailed_uncurtailed scope     variable                      
Central                  Curtailed             Capture   Fixed solar PV          NaN   
                                                         Floating offshore wind  NaN   
                                                         Offshore wind           NaN   
                                                         Onshore wind            NaN   
                                                         Tracking solar PV       NaN   
...                                                                              ...   
Lowextendedgenerationtax N/A                   Commodity Gas price               NaN   
                         Uncurtailed           Capture   Fixed solar PV          NaN   
                                                         Floating offshore wind  NaN   
                                                         Onshore wind            NaN   
                                                         Tracking solar PV       NaN   

region                                                                          Germany  \
Scenario                 curtailed_uncurtailed scope     variable                         
Central                  Curtailed             Capture   Fixed solar PV             NaN   
                                                         Floating offshore wind     NaN   
                                                         Offshore wind              NaN   
                                                         Onshore wind               NaN   
                                                         Tracking solar PV          NaN   
...                                                                                 ...   
Lowextendedgenerationtax N/A                   Commodity Gas price                  NaN   
                         Uncurtailed           Capture   Fixed solar PV             NaN   
                                                         Floating offshore wind     NaN   
                                                         Onshore wind               NaN   
                                                         Tracking solar PV          NaN   

region                                                                            Ireland  \
Scenario                 curtailed_uncurtailed scope     variable                           
Central                  Curtailed             Capture   Fixed solar PV          2026-Jan   
                                                         Floating offshore wind       NaN   
                                                         Offshore wind           2026-Jan

In [21]:
# Load the SharePoint base path from the .env file so that the path is never hardcoded in code.
# SP_BASE_PATH should point to the root of the OneDrive-synced SharePoint library.
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()
print(os.environ.get('SP_BASE_PATH'))

path = Path(os.getenv('SP_BASE_PATH')) / 'General/Market Research/Trackers - EUROPE/Prices/Backup/Aurora/Python'

tracker_name = 'prices_tracker.xlsx'
driver_tracker_name = 'prices_tracker_drivers.xlsx'


df_concat_fil_pivotted.to_excel(f'{path}/{tracker_name}', index=False)

# We export drivers dict, so that each value of the dict is one sheet in the excel file,
# allowing the tracker to have one sheet per release with the specific driver values for that release.

with pd.ExcelWriter(f'{path}/{driver_tracker_name}') as writer:
    for release_string, df_drivers_pivot in release_dict.items():
        df_drivers_pivot.to_excel(writer, sheet_name=release_string, index=True)

# # Optionally verify the upload by reading back a previously saved file
# df_try = pd.read_excel(f'{path}/{tracker_name}')

# df_try['release_string'].unique()

C:/Users/mpy/OneDrive - Exus Management Partners/EU - Strategy & Markets - Documentos


In [22]:
# # Authenticate with Microsoft Graph and upload the tracker Excel to SharePoint.
# # Credentials are loaded from YAML config + environment variables — never hardcoded.
# from pathlib import Path
# from common_libs.sp.settings import load_ms_config
# from common_libs.sp.graph_client import GraphSharePointClient
# import os

# # Load config (YAML + env secret) and build the SharePointConfig dataclass
# _, sp_cfg = load_ms_config()

# client = GraphSharePointClient(sp_cfg)

# # Extract just the filename from the local tracker path for use as the SharePoint upload target
# tracker_path = Path(tracker_path).name

# sharepoint_folder = (
#     "EU - Strategy & Markets - DocumentosGeneral/"
#     "Market Research/Trackers - EUROPE/Prices/Backup"
# )

# sharepoint_target_path = f"{sharepoint_folder}/{tracker_path}"

# # Upload (create or overwrite) the tracker; raises RuntimeError if the MSAL token is expired
# item = client.upload_file(str(tracker_path), sharepoint_target_path)

# print("Uploaded:", item.get("name"))
# print("Link:", item.get("webUrl"))